# SSM Explained: Mamba-2 from First Principles

Understand how structured state space models work in
hybrid architectures. Compare SSM's recurrent state
to attention's explicit memory.

**Prerequisites:** `uv sync --extra dev`

In [ ]:
import mlx.core as mx

from lmxlab.models.base import LanguageModel
from lmxlab.models.falcon import falcon_h1_tiny

## What is an SSM?

A **Structured State Space Model** (S4;
[Gu et al., 2021](https://arxiv.org/abs/2111.00396)) maintains
a hidden state `h` that evolves as it processes each token:

```
h_t = A * h_{t-1} + B * x_t    (state update)
y_t = C * h_t + D * x_t         (output)
```

Unlike attention (which looks at ALL previous tokens each step),
SSMs compress history into a fixed-size state. This gives:
- **O(n)** vs O(n^2) complexity
- **Fixed memory** regardless of sequence length
- But potentially **lossy compression** of long-range info

## Mamba-2: Selective State Spaces

Mamba ([Gu & Dao, 2023](https://arxiv.org/abs/2312.00752)) makes
B, C, and the discretization step Δ **data-dependent**
(selective): the model learns WHAT to remember based on the
current input, rather than applying a fixed filter. (A is kept
input-independent; see Section 3.5.2 of the paper.) Mamba-2
([Dao & Gu, 2024](https://arxiv.org/abs/2405.21060)) reformulates
this through a state space duality with attention, enabling
hardware-efficient chunked computation.

Key parameters in lmxlab's `BlockConfig`:
- `mamba_n_heads`: Number of SSM heads
- `mamba_head_dim`: Dimension per head
- `ssm_state_size`: Hidden state size (memory capacity)
- `mamba_expand`: Expansion factor for inner dimension
- `mamba_chunk_size`: Chunk size for efficient computation

In [ ]:
# Inspect a hybrid model's layer configuration
cfg = falcon_h1_tiny()
print(f"Falcon-H1 tiny: {cfg.n_layers} layers\n")

for i in range(cfg.n_layers):
    bcfg = cfg.get_block_config(i)
    if "mamba" in bcfg.attention:
        print(f"Layer {i}: Mamba-2 SSM")
        print(f"  n_heads={bcfg.mamba_n_heads}, head_dim={bcfg.mamba_head_dim}")
        print(f"  state_size={bcfg.ssm_state_size}, expand={bcfg.mamba_expand}")
        inner = bcfg.d_model * bcfg.mamba_expand
        print(f"  inner_dim={inner}")
    else:
        print(f"Layer {i}: GQA Attention")
        print(f"  n_heads={bcfg.n_heads}, n_kv_heads={bcfg.effective_n_kv_heads}")

## SSM vs Attention: Forward Pass

Both process the same input but through different mechanisms.
Let's verify they produce valid outputs.

In [ ]:
model = LanguageModel(cfg)
mx.eval(model.parameters())

tokens = mx.zeros((1, 32), dtype=mx.int32)
logits, caches = model(tokens)
mx.eval(logits)

print(f"Input shape:  {tokens.shape}")
print(f"Output shape: {logits.shape}")
print(f"Params: {model.count_parameters():,}")
print(f"\nCache types per layer:")
for i, cache in enumerate(caches):
    bcfg = cfg.get_block_config(i)
    layer_type = "SSM" if "mamba" in bcfg.attention else "Attn"
    if cache is None:
        print(f"  Layer {i} ({layer_type}): None")
    elif isinstance(cache, tuple):
        shapes = [c.shape if hasattr(c, 'shape') else type(c) for c in cache]
        print(f"  Layer {i} ({layer_type}): {shapes}")

## Why Hybrid?

Pure SSM models struggle with tasks requiring precise recall of
specific past tokens (see Section 4.1 in
[Gu & Dao, 2023](https://arxiv.org/abs/2312.00752) for selective
copying and induction head analysis). Attention excels at this
but is expensive for long sequences.

**Hybrid architectures** combine both:
- SSM layers handle sequential patterns efficiently
- Attention layers provide precise token-level recall
- Typical ratio: 3:1 to 7:1 SSM:attention
  ([Lieber et al., 2024](https://arxiv.org/abs/2403.19887))

The `MMM*` pattern in Falcon-H1
([Zuo et al., 2025](https://arxiv.org/abs/2507.22448)) and Bamba
([IBM et al., 2025](https://huggingface.co/blog/bamba)) means:
- `M` = Mamba-2 SSM block
- `*` = GQA attention block
- Pattern repeats across the model

## References

- Gu et al. (2021). [Efficiently Modeling Long Sequences with Structured State Spaces](https://arxiv.org/abs/2111.00396). ICLR 2022.
- Gu & Dao (2023). [Mamba: Linear-Time Sequence Modeling with Selective State Spaces](https://arxiv.org/abs/2312.00752). COLM 2024.
- Dao & Gu (2024). [Transformers are SSMs: Generalized Models and Efficient Algorithms Through Structured State Space Duality](https://arxiv.org/abs/2405.21060). ICML 2024.
- Lieber et al. (2024). [Jamba: A Hybrid Transformer-Mamba Language Model](https://arxiv.org/abs/2403.19887). arXiv preprint.
- IBM et al. (2025). [Bamba: Inference-Efficient Hybrid Mamba2 Model](https://huggingface.co/blog/bamba). HuggingFace blog post.
- Zuo et al. (2025). [Falcon-H1: A Family of Hybrid-Head Language Models Redefining Efficiency and Performance](https://arxiv.org/abs/2507.22448). arXiv preprint.